
# Get Started with Databrick


In [0]:
%sql
SELECT * FROM workspace.default.wine_quality LIMIT 10

## Define catalog & schema space

In [0]:
%sql
USE CATALOG 'fashion_trends_based_on_fashion_products_and_outfit_combinations';
-- USE SCHEMA  'default';
SELECT current_catalog(), current_schema();
SELECT * FROM fashion_products LIMIT 100

In [0]:
%sql
USE CATALOG 'workspace';
DESCRIBE SCHEMA EXTENDED default; --describe the schema metadata
SHOW tables in default; -- display the table in the current schema (under this catalog)
SHOW volumes in default

#### To query a volume file (say csv), you can use below code (note that xlsx file is not supported, due to multiple tabs it supports)

In [0]:
spark.sql("""
SELECT *
FROM read_files(
  '/Volumes/workspace/default/test_volume/test_directory/synthetic_user_query_data.csv',
  format => 'csv',
  header => true,
  inferSchema => true
)
""").display()

In [0]:
%sql
-- Another way to to read a csv volumne file in SQL directly (assuming your csv first row is column name)
SELECT *
FROM read_files(
  '/Volumes/workspace/default/test_volume/test_directory/synthetic_user_query_data.csv',
  format => 'csv',
  header => true,
  inferSchema => true -- 
)

### Create the delta table from a volume csv file using SQL

In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.user_query;

-- Create the delta table using csv 
CREATE TABLE workspace.default.user_query AS
SELECT * FROM read_files(
  '/Volumes/workspace/default/test_volume/test_directory/synthetic_user_query_data.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);
    
-- Display the table
SELECT * FROM workspace.default.user_query

### In python, you can also read the volume csv file directly and create a spark dataframe

option('inferschema','true') to let the system to auto-detect each column type rather than treating every column as a string


In [0]:
# read the csv file aand create the spark dataframe
sdf = spark.read.format('csv').option('header','true').option('inferschema','true').load('/Volumes/workspace/default/test_volume/test_directory/synthetic_user_query_data.csv')

# create the delta table from the spark dataframe
sdf.write.format('delta').mode('overwrite').saveAsTable('workspace.default.user_query')


In [0]:
# Read the created delta table using python
spark.read.table('workspace.default.user_query').display()

# show the tables in the current schema
spark.catalog.listTables()

### Get table metadata

In [0]:
%sql
DESCRIBE DETAIL user_query -- get the metadata of the tale (e.g., format, location)

In [0]:
%sql
DESCRIBE EXTENDED user_query -- get the column types and metadata of the table

### Time travel of history of table

In [0]:
%sql
DESCRIBE HISTORY user_query -- get the history of the table
-- note the operationMetrics column which is a json file that explains what actions have been done to the table

### Insert, Update, Delete ercords in Delta Table

In [0]:
%sql
select * from user_query

In [0]:
%sql
-- drop the _rescued_data column
ALTER TABLE user_query SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name'); -- Enable column mapping mode 'name' on your table
ALTER TABLE user_query DROP COLUMN _rescued_data;
select * from user_query

In [0]:
%sql
-- Insert
INSERT INTO user_query
VALUES
  ('window broken', '12.1' );

In [0]:
%sql
-- update
UPDATE user_query 
  SET search_result = '20.2' 
  WHERE user_query = 'window broken';
    
select * from user_query


In [0]:
%sql
-- delete
DELETE FROM user_query 
  WHERE user_query = 'window broken';
select * from user_query

In [0]:
%sql
describe history user_query

### Time travel to see previous version

In [0]:
%sql
select *
from user_query@v8 
-- alternatively: from user_query version as of 8 

### To see the original table, simply check the version 0 when it is created


In [0]:
%sql
SELECT * FROM user_query@v0

### Drop table

In [0]:
%sql
-- DROP TABLE IF EXISTS user_query 

## How to ingest data into delta lake 
- Create table
- Upload UI
- Copy Info (incrementally load data from a file location into a Delta table)
- Autoloader

In [0]:
# check your current space through spark python
spark.catalog.setCurrentCatalog('workspace')
spark.catalog.setCurrentDatabase('default')

# display current database tables
spark.catalog.listTables()

In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.user_query;

-- Create the delta table using csv 
CREATE TABLE workspace.default.user_query AS
SELECT * FROM read_files(
  '/Volumes/workspace/default/test_volume/test_directory/synthetic_user_query_data.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);
    
-- Display the table
SELECT * FROM workspace.default.user_query

#### Create the table using COPY INTO
Note that we are loading the data from a volume directory as one delta table (if the volume directory has multiple csv files, this COPY INTO will consolidate all the files into a delta table with the schema)

As new files arrive in the volume directory, rerun the COPY INTO will only do the incremental load (as the previous files have already been loaded)

In [0]:
%sql
-- Create a empty table for holding user_query
DROP TABLE IF EXISTS user_query_copy;

CREATE TABLE user_query_copy (
  user_query STRING,
  search_result STRING
);


In [0]:
%sql
COPY INTO user_query_copy
  FROM '/Volumes/workspace/default/test_volume/test_directory/'
  FILEFORMAT = CSV -- specify the volume files that we are looking for are csv file only
  Format_options (
    'header' = 'TRUE',
    'inferSchema' = 'TRUE'
);
select * from user_query_copy

In [0]:
# doing the same thing using python sql
# the number of rows affected is zero as the table is already loaded (Idempotent Operations)
spark.sql("""
COPY INTO user_query_copy
  FROM '/Volumes/workspace/default/test_volume/test_directory/'
  FILEFORMAT = CSV
  Format_options (
    'header' = 'TRUE',
    'inferSchema' = 'TRUE'
  )
""").display()

In [0]:
%sql
DESCRIBE HISTORY user_query_copy


In [0]:
%sql -- clean the duplicate
DROP TABLE IF EXISTS user_query_copy;
Show tables

## Data Transformation